# Автоматический подбор гиперпараметров для классификаторов и применение LSA к предобработке тектсов

## 1. Загрузка необходимых библиотек и данных

In [31]:
import re
import pandas as pd
from datasets import load_dataset

from sklearn.datasets import fetch_20newsgroups

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.tree import DecisionTreeClassifier
from  sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

from sklearn.metrics import f1_score

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\a

True

### 1.1 Загрузка датасета emotion

In [32]:
dataset_emotion_train = load_dataset('emotion', split='train')
dataset_emotion_test = load_dataset('emotion', split='test')

print(f"emotion train size: {len(dataset_emotion_train)}")
print(f"emotion test size: {len(dataset_emotion_test)}")

emotion train size: 16000
emotion test size: 2000


### 1.2 Загрузка датачета 20newsgroups(только указанные  категории)

In [33]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

dataset_news_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
dataset_news_test = fetch_20newsgroups(subset='test', categories=categories,shuffle=True, random_state=42)

print(f"news train size: {len(dataset_news_train.data)}")
print(f"news test size: {len(dataset_news_test.data)}")

news train size: 2345
news test size: 1561


## 2. Предобработка текстов

### 2.1 Удаление шума(для news)

In [34]:
def clean_noise(text):
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum/len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text= re.sub(r'\S+@\S+', '', text)
    text = re.sub(r"[^\w\s.!?']", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 2.2 Токенизация, стемминг, лемматизация, лемматизация(только сущ и прил), LSA

In [35]:
def get_wordnet_pos(tag):
    """Преобразует тег POS из NLTK в формат WordNet."""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [36]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

def stem_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

def lemmatize_text(text):
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    lemmarized_words = []
    for word, tag in pos_tags:
        if word not in stop_words:
            wordnet_tag = get_wordnet_pos(tag)
            lemmarized_words.append(lemmatizer.lemmatize(word, pos=wordnet_tag))
    return ' '.join(lemmarized_words)

def lemmatize_noun_adj_text(text):
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    lemmarized_words = []
    for word, tag in pos_tags:
        if word not in stop_words:
            if tag.startswith('NN') or tag.startswith('J'):
                wordnet_tag = get_wordnet_pos(tag)
                lemmarized_words.append(lemmatizer.lemmatize(word, pos=wordnet_tag))
    return ' '.join(lemmarized_words)

In [37]:
preprocessors = {
    'tokenization': tokenize_text,
    'stemming': stem_text,
    'lemmatization': lemmatize_text,
    'lemmatization_noun_adj': lemmatize_noun_adj_text
}

### 2.3 Применение предобработки

In [38]:
emotion_train_texts = dataset_emotion_train['text']
emotion_train_labels = dataset_emotion_train['label']
emotion_test_texts = dataset_emotion_test['text']
emotion_test_labels = dataset_emotion_test['label']

In [39]:
news_train_texts = [clean_noise(text) for text in dataset_news_train.data]
news_test_texts = [clean_noise(text) for text in dataset_news_test.data]
news_train_labels = dataset_news_train.target
news_test_labels = dataset_news_test.target

In [40]:
processed_train = {'emotion': {}, 'news': {}}
processed_test = {'emotion': {}, 'news': {}}

In [41]:
for prep_name, prep_func in preprocessors.items():
    print(f"Processing emotion with {prep_name}...")
    processed_train['emotion'][prep_name] = [prep_func(text) for text in emotion_train_texts]
    processed_test['emotion'][prep_name] = [prep_func(text) for text in emotion_test_texts]

Processing emotion with tokenization...
Processing emotion with stemming...
Processing emotion with lemmatization...
Processing emotion with lemmatization_noun_adj...


In [42]:
for prep_name, prep_func in preprocessors.items():
    print(f"Processing emotion with {prep_name}...")
    processed_train['news'][prep_name] = [prep_func(text) for text in news_train_texts]
    processed_test['news'][prep_name] = [prep_func(text) for text in news_test_texts]

Processing emotion with tokenization...
Processing emotion with stemming...
Processing emotion with lemmatization...
Processing emotion with lemmatization_noun_adj...


## 3. Векторизаторы и классификаторы

In [85]:
vectorizers = {
    'binary': CountVectorizer(binary=True),
    'count': CountVectorizer(binary=False),
    'tfidf': TfidfVectorizer()
}

classifiers = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'AdaBoostClassifier': AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=42),random_state=42)
}

In [44]:
results = []
dataset_names = ['emotion', 'news']

## 4. Цикл экспериментов (по всем датасетам, предобработкам, векторизаторам и классификаторам)

In [45]:
for dataset_name in dataset_names:
    print(f"\n=============== ДАТАСЕТ:{dataset_name.upper()} ==============")
    y_train = emotion_train_labels if dataset_name == 'emotion' else news_train_labels
    y_test = emotion_test_labels if dataset_name == 'emotion' else news_test_labels

    for prep_name in preprocessors.keys():
        X_train_prep = processed_train[dataset_name][prep_name]
        X_test_prep = processed_test[dataset_name][prep_name]

        for vec_name, vectorizer in vectorizers.items():
            print(f"\nПредобработка: {prep_name}, Векторизация: {vec_name}")
            X_train_vec = vectorizer.fit_transform(X_train_prep)
            X_test_vec = vectorizer.transform(X_test_prep)

            for clf_name, clf in classifiers.items():
                clf.fit(X_train_vec, y_train)
                y_pred = clf.predict(X_test_vec)

                f1_micro = f1_score(y_test, y_pred, average='micro')
                f1_macro = f1_score(y_test, y_pred, average='macro')
                f1_weighted = f1_score(y_test, y_pred, average='weighted')

                results.append({
                    'Dataset': dataset_name,
                    'Preprocessing': prep_name,
                    'Vectorizer': vec_name,
                    'Classifier': clf_name,
                    'F1_micro': f1_micro,
                    'F1_macro': f1_macro,
                    'F1_weighted': f1_weighted
                })
                print(f"{clf_name}: micro={f1_micro:.4f}, macro={f1_macro:.4f}, weighted={f1_weighted:.4f}")


=============== ДАТАСЕТ:EMOTION ==============

Предобработка: tokenization, Векторизация: binary
DecisionTreeClassifier: micro=0.8680, macro=0.8070, weighted=0.8687
RandomForestClassifier: micro=0.8835, macro=0.8251, weighted=0.8830
GradientBoostingClassifier: micro=0.8445, macro=0.8001, weighted=0.8452
AdaBoostClassifier: micro=0.3480, macro=0.0900, weighted=0.1867

Предобработка: tokenization, Векторизация: count
DecisionTreeClassifier: micro=0.8675, macro=0.8039, weighted=0.8684
RandomForestClassifier: micro=0.8870, macro=0.8262, weighted=0.8867
GradientBoostingClassifier: micro=0.8440, macro=0.8002, weighted=0.8445
AdaBoostClassifier: micro=0.3480, macro=0.0900, weighted=0.1867

Предобработка: tokenization, Векторизация: tfidf
DecisionTreeClassifier: micro=0.8630, macro=0.8062, weighted=0.8635
RandomForestClassifier: micro=0.8855, macro=0.8211, weighted=0.8838
GradientBoostingClassifier: micro=0.8440, macro=0.8003, weighted=0.8441
AdaBoostClassifier: micro=0.3480, macro=0.0896, w

## 5. Таблица результатов

In [46]:
from IPython.display import display

In [47]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='F1_micro', ascending=False)

In [48]:
for dataset_name in dataset_names:
    for clf_name, _ in classifiers.items():
        display(df_results[(df_results['Dataset'] == dataset_name) & (df_results['Classifier'] == clf_name)].head(3))

,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
0,emotion,tokenization,binary,DecisionTreeClassifier,0.8680,0.807011,0.868746
4,emotion,tokenization,count,DecisionTreeClassifier,0.8675,0.803927,0.868418
8,emotion,tokenization,tfidf,DecisionTreeClassifier,0.8630,0.806199,0.863499


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
5,emotion,tokenization,count,RandomForestClassifier,0.8870,0.826223,0.886735
9,emotion,tokenization,tfidf,RandomForestClassifier,0.8855,0.821095,0.883758
1,emotion,tokenization,binary,RandomForestClassifier,0.8835,0.825056,0.883021


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
2,emotion,tokenization,binary,GradientBoostingClassifier,0.8445,0.800090,0.845171
6,emotion,tokenization,count,GradientBoostingClassifier,0.8440,0.800222,0.844545
10,emotion,tokenization,tfidf,GradientBoostingClassifier,0.8440,0.800254,0.844057


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
23,emotion,stemming,tfidf,AdaBoostClassifier,0.3545,0.104117,0.195434
19,emotion,stemming,count,AdaBoostClassifier,0.3540,0.104155,0.195514
15,emotion,stemming,binary,AdaBoostClassifier,0.3540,0.104155,0.195514


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
84,news,lemmatization_noun_adj,binary,DecisionTreeClassifier,0.625881,0.626835,0.626725
92,news,lemmatization_noun_adj,tfidf,DecisionTreeClassifier,0.616272,0.617648,0.617476
56,news,tokenization,tfidf,DecisionTreeClassifier,0.616272,0.616957,0.616692


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
77,news,lemmatization,count,RandomForestClassifier,0.772582,0.772862,0.772803
61,news,stemming,binary,RandomForestClassifier,0.768097,0.768236,0.768245
65,news,stemming,count,RandomForestClassifier,0.768097,0.768134,0.768109


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
62,news,stemming,binary,GradientBoostingClassifier,0.763613,0.766636,0.766565
66,news,stemming,count,GradientBoostingClassifier,0.760410,0.763811,0.763689
74,news,lemmatization,binary,GradientBoostingClassifier,0.755285,0.758912,0.758826


,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted
75,news,lemmatization,binary,AdaBoostClassifier,0.598334,0.604689,0.604570
83,news,lemmatization,tfidf,AdaBoostClassifier,0.592569,0.599234,0.599410
71,news,stemming,tfidf,AdaBoostClassifier,0.591928,0.598137,0.597961


### Лучшие комбинации

In [87]:
best_for_emotion = {
    'DecisionTreeClassifier': {
        'preprocessing': 'tokenization',
        'vectorizer': 'binary'
    },
    'RandomForestClassifier': {
        'preprocessing': 'tokenization',
        'vectorizer': 'count'
    },
    'GradientBoostingClassifier': {
        'preprocessing': 'tokenization',
        'vectorizer': 'binary'
    },
    'AdaBoostClassifier': {
        'preprocessing': 'stemming',
        'vectorizer': 'tfidf'
    }
}

In [79]:
best_for_news = {
    'DecisionTreeClassifier': {
        'preprocessing': 'lemmatization_noun_adj',
        'vectorizer': 'binary'
    },
    'RandomForestClassifier': {
        'preprocessing': 'lemmatization',
        'vectorizer': 'count'
    },
    'GradientBoostingClassifier': {
        'preprocessing': 'stemming',
        'vectorizer': 'binary'
    },
    'AdaBoostClassifier': {
        'preprocessing': 'lemmatization',
        'vectorizer': 'binary'
    }
}

## 6. Подбор гиперпараметров

### 6.1 Сетки параметров

In [109]:
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', None],
    'min_impurity_decrease': [0.0, 0.01]
}

In [131]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

In [107]:
param_grid_gb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'subsample': [0.8, 1.0]
}

In [110]:
param_grid_ada = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.5, 1.0, 1.5],
    'estimator__max_depth': [1, 2, 3],
    'estimator__min_samples_split': [2, 5],
    'estimator__min_samples_leaf': [1, 2]
}

In [132]:
param_grid = {
    'DecisionTreeClassifier': param_grid_dt,
    'RandomForestClassifier': param_grid_rf,
    'GradientBoostingClassifier': param_grid_gb,
    'AdaBoostClassifier': param_grid_ada
}

In [56]:
result_optimized_hyper = []

### 6.2 Функция подбора параметров

In [113]:
from sklearn.model_selection import RandomizedSearchCV

In [129]:
def tune_classifier(dataset_name, clf_name, prep_name,vec_name, param_grid):
    X_train = processed_train[dataset_name][prep_name]
    X_test = processed_test[dataset_name][prep_name]
    y_train = emotion_train_labels if dataset_name == 'emotion' else news_train_labels
    y_test = emotion_test_labels if dataset_name == 'emotion' else news_test_labels

    if vec_name == 'binary':
        vec = CountVectorizer(binary=True)
    elif vec_name == 'count':
        vec = CountVectorizer(binary=False)
    else:
        vec = TfidfVectorizer()

    X_train_vec = vec.fit_transform(X_train)
    X_test_vec = vec.transform(X_test)

    # Поиск гиперпараметров
    search = RandomizedSearchCV(
        classifiers[clf_name], param_distributions=param_grid, n_iter=20, scoring = 'f1_micro', random_state=42, cv=3, n_jobs=-1
    )
    search.fit(X_train_vec,y_train)
    best_clf = search.best_estimator_
    y_pred = best_clf.predict(X_test_vec)
    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    result_optimized_hyper.append({
        'Dataset': dataset_name,
        'Preprocessing': prep_name,
        'Vectorizer': vec_name,
        'Classifier': clf_name,
        'F1_micro': f1_micro,
        'F1_macro': f1_macro,
        'F1_weighted': f1_weighted,
        'Best_params': str(search.best_params_)
    })

    print(f"{dataset_name} | {prep_name} | {vec_name} | {clf_name} tuned: micro={f1_micro:.4f}")

In [133]:
for clf_name, pipe in best_for_emotion.items():
    tune_classifier('emotion',clf_name, pipe['preprocessing'], pipe['vectorizer'], param_grid[clf_name])

emotion | tokenization | binary | DecisionTreeClassifier tuned: micro=0.7730
emotion | tokenization | count | RandomForestClassifier tuned: micro=0.8940
emotion | tokenization | binary | GradientBoostingClassifier tuned: micro=0.8775
emotion | stemming | tfidf | AdaBoostClassifier tuned: micro=0.3810


In [134]:
for clf_name, pipe in best_for_news.items():
    tune_classifier('news',clf_name, pipe['preprocessing'], pipe['vectorizer'], param_grid[clf_name])

news | lemmatization_noun_adj | binary | DecisionTreeClassifier tuned: micro=0.5490
news | lemmatization | count | RandomForestClassifier tuned: micro=0.7623
news | stemming | binary | GradientBoostingClassifier tuned: micro=0.7771
news | lemmatization | binary | AdaBoostClassifier tuned: micro=0.7329


In [135]:
df_result_optimized_hyper = pd.DataFrame(result_optimized_hyper)
df_result_optimized_hyper = df_result_optimized_hyper.sort_values(by='F1_micro', ascending=False)
display(df_result_optimized_hyper)

,Dataset,Preprocessing,Vectorizer,Classifier,F1_micro,F1_macro,F1_weighted,Best_params
31,emotion,tokenization,count,RandomForestClassifier,0.894000,0.836105,0.891889,"{'n_estimators': 100, 'min_samples_split': 10,..."
5,emotion,tokenization,binary,GradientBoostingClassifier,0.883500,0.836455,0.885052,"{'n_estimators': 100, 'min_samples_split': 5, ..."
10,emotion,tokenization,binary,GradientBoostingClassifier,0.883500,0.836455,0.885052,"{'n_estimators': 100, 'min_samples_split': 5, ..."
14,emotion,tokenization,binary,GradientBoostingClassifier,0.883500,0.836455,0.885052,"{'n_estimators': 100, 'min_samples_split': 5, ..."
2,emotion,tokenization,binary,GradientBoostingClassifier,0.883500,0.836455,0.885052,"{'n_estimators': 100, 'min_samples_split': 5, ..."
32,emotion,tokenization,binary,GradientBoostingClassifier,0.877500,0.832051,0.878433,"{'subsample': 0.8, 'n_estimators': 100, 'min_s..."
18,emotion,stemming,binary,GradientBoostingClassifier,0.854500,0.803753,0.855635,"{'n_estimators': 100, 'min_samples_split': 5, ..."
36,news,stemming,binary,GradientBoostingClassifier,0.777066,0.778927,0.778752,"{'subsample': 0.8, 'n_estimators': 100, 'min_s..."
24,emotion,tokenization,binary,DecisionTreeClassifier,0.773000,0.735353,0.771289,"{'min_samples_split': 10, 'min_samples_leaf': ..."
29,emotion,tokenization,binary,DecisionTreeClassifier,0.773000,0.735353,0.771289,"{'min_samples_split': 10, 'min_samples_leaf': ..."


### 6.3 Выводы

1) Для датасета emotion

- RandomForestClassifier: после подбора микро‑F1 вырос с 0,887 до 0,894

- GradientBoostingClassifier: значительный рост с 0,8445 до 0,8835

- DecisionTreeClassifier: ухудшение с 0,868 до 0,773 – возможно, сетка параметров была недостаточно проработана

- AdaBoostClassifier: микро‑F1 вырос с 0,3545 до 0,381, но всё ещё остаётся крайне низким - эта модель плохо справляется с задачей классификации эмоций

2) Для датасета news

- RandomForestClassifier: результат практически не изменился: с 0,7726 до 0,7623)

- GradientBoostingClassifier: улучшение с 0,7636 до 0,7771 - подтверждает, что бустинговые модели выигрывают от тонкой настройки

- DecisionTreeClassifier: сильное падение с 0,6259 до 0,5490

- AdaBoostClassifier: заметный рост с 0,5983 до 0,7329 - подбор существенно повысил

3) Общие мысли:
Причинами ухудшения могут быть:

 - ограниченное число итераций случайного поиска (n_iter=20),

 - использование 3‑кратной кросс‑валидации

- неполное покрытие пространства гиперпараметров

### 7. LSA

In [136]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD

In [137]:
components = [50,100,200,300,500]

In [138]:
results_LSA = []

In [139]:
def run_lsa(dataset_name,prep_name,clf_name,components_list):
    X_train = processed_train[dataset_name][prep_name]
    X_test = processed_test[dataset_name][prep_name]
    y_train = emotion_train_labels if dataset_name == 'emotion' else news_train_labels
    y_test = emotion_test_labels if dataset_name == 'emotion' else news_test_labels

    # Без LSA
    pipe_base = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', classifiers[clf_name])
    ])

    pipe_base.fit(X_train,y_train)
    y_pred_base = pipe_base.predict(X_test)
    f1_micro_base = f1_score(y_test, y_pred_base, average='micro')
    f1_macro_base = f1_score(y_test, y_pred_base, average='macro')
    f1_weighted_base = f1_score(y_test, y_pred_base, average='weighted')

    results_LSA.append({
        'Dataset': dataset_name,
        'Preprocessing': prep_name,
        'Vectorizer': 'tf-idf',
        'Classifier': clf_name,
        'F1_micro': f1_micro_base,
        'F1_macro': f1_macro_base,
        'F1_weighted': f1_weighted_base,
    })

    print(f"{dataset_name} | {prep_name} | {clf_name} |micro={f1_micro_base:.4f}")

    for n in components_list:
        pipe_lsa = Pipeline([
            ('tfidf', TfidfVectorizer()),
            ('lsa', TruncatedSVD(n_components=n, random_state=42)),
            ('clf', classifiers[clf_name])
        ])

        pipe_lsa.fit(X_train,y_train)
        y_pred_lsa = pipe_lsa.predict(X_test)
        f1_micro_lsa = f1_score(y_test, y_pred_lsa, average='micro')
        f1_macro_lsa = f1_score(y_test, y_pred_lsa, average='macro')
        f1_weighted_lsa = f1_score(y_test, y_pred_lsa, average='weighted')

        results_LSA.append({
            'Dataset': dataset_name,
            'Preprocessing': prep_name,
            'Vectorizer': f"LSA_{n}",
            'Classifier': clf_name,
            'F1_micro': f1_micro_lsa,
            'F1_macro': f1_macro_lsa,
            'F1_weighted': f1_weighted_lsa,
        })

        print(f"  n={n}: micro={f1_micro_lsa:.4f}")

In [140]:
best_for_emotion_without_GBM = {
     'DecisionTreeClassifier': {
        'preprocessing': 'tokenization',
        'vectorizer': 'binary'
    },
    'RandomForestClassifier': {
        'preprocessing': 'tokenization',
        'vectorizer': 'count'
    },
    'AdaBoostClassifier': {
        'preprocessing': 'stemming',
        'vectorizer': 'tfidf'
    }
}

In [141]:
best_for_news_without_GBM = {
    'DecisionTreeClassifier': {
        'preprocessing': 'lemmatization_noun_adj',
        'vectorizer': 'binary'
    },
    'RandomForestClassifier': {
        'preprocessing': 'lemmatization',
        'vectorizer': 'count'
    },
    'AdaBoostClassifier': {
        'preprocessing': 'lemmatization',
        'vectorizer': 'binary'
    }
}

In [142]:
for clf_name, pipe in best_for_emotion_without_GBM.items():
    run_lsa('emotion', pipe['preprocessing'], clf_name, components)

emotion | tokenization | DecisionTreeClassifier |micro=0.8630
  n=50: micro=0.3060
  n=100: micro=0.3520
  n=200: micro=0.3955
  n=300: micro=0.3880
  n=500: micro=0.4315
emotion | tokenization | RandomForestClassifier |micro=0.8855
  n=50: micro=0.4690
  n=100: micro=0.5335
  n=200: micro=0.5900
  n=300: micro=0.6340
  n=500: micro=0.6560
emotion | stemming | AdaBoostClassifier |micro=0.7340
  n=50: micro=0.3500
  n=100: micro=0.4075
  n=200: micro=0.4865
  n=300: micro=0.4940
  n=500: micro=0.5065


In [143]:
for clf_name, pipe in best_for_news_without_GBM.items():
    run_lsa('news', pipe['preprocessing'], clf_name, components)

news | lemmatization_noun_adj | DecisionTreeClassifier |micro=0.6163
  n=50: micro=0.5990
  n=100: micro=0.6092
  n=200: micro=0.6163
  n=300: micro=0.6137
  n=500: micro=0.6182
news | lemmatization | RandomForestClassifier |micro=0.7675
  n=50: micro=0.7290
  n=100: micro=0.7316
  n=200: micro=0.7348
  n=300: micro=0.7380
  n=500: micro=0.7309
news | lemmatization | AdaBoostClassifier |micro=0.6534
  n=50: micro=0.6400
  n=100: micro=0.6586
  n=200: micro=0.6618
  n=300: micro=0.6528
  n=500: micro=0.6534


In [144]:
df_results_LSA = pd.DataFrame(results_LSA)
df_results_LSA.sort_values(by='F1_micro', ascending=False)
display(results_LSA)

[{'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'tf-idf',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.863,
  'F1_macro': 0.8061988502341788,
  'F1_weighted': 0.863498855490817},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_50',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.306,
  'F1_macro': 0.2221763453272557,
  'F1_weighted': 0.307422497618017},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_100',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.352,
  'F1_macro': 0.2641248112985211,
  'F1_weighted': 0.3556936975581032},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_200',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.3955,
  'F1_macro': 0.3048929316628195,
  'F1_weighted': 0.39832857921695636},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_300',
  'Classifier': 'DecisionTreeCla

## 8. LSA + только сущ и прил

In [145]:
prep_noun_adj = 'lemmatization_noun_adj'

In [146]:
for clf_name, pipe in best_for_emotion_without_GBM.items():
    run_lsa('emotion', prep_noun_adj, clf_name, components)

emotion | lemmatization_noun_adj | DecisionTreeClassifier |micro=0.6780
  n=50: micro=0.3805
  n=100: micro=0.4070
  n=200: micro=0.4395
  n=300: micro=0.4670
  n=500: micro=0.4705
emotion | lemmatization_noun_adj | RandomForestClassifier |micro=0.7120
  n=50: micro=0.5120
  n=100: micro=0.5480
  n=200: micro=0.6050
  n=300: micro=0.6155
  n=500: micro=0.6205
emotion | lemmatization_noun_adj | AdaBoostClassifier |micro=0.6195
  n=50: micro=0.4785
  n=100: micro=0.5060
  n=200: micro=0.5420
  n=300: micro=0.5560
  n=500: micro=0.5565


In [147]:
for clf_name, pipe in best_for_news_without_GBM.items():
    run_lsa('news', prep_noun_adj, clf_name, components)

news | lemmatization_noun_adj | DecisionTreeClassifier |micro=0.6163
  n=50: micro=0.5990
  n=100: micro=0.6092
  n=200: micro=0.6163
  n=300: micro=0.6137
  n=500: micro=0.6182
news | lemmatization_noun_adj | RandomForestClassifier |micro=0.7438
  n=50: micro=0.7156
  n=100: micro=0.7175
  n=200: micro=0.7098
  n=300: micro=0.7149
  n=500: micro=0.7021
news | lemmatization_noun_adj | AdaBoostClassifier |micro=0.6413
  n=50: micro=0.6534
  n=100: micro=0.6233
  n=200: micro=0.6323
  n=300: micro=0.6374
  n=500: micro=0.6451


In [148]:
df_results_LSA.sort_values(by='F1_micro', ascending=False)
display(results_LSA)

[{'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'tf-idf',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.863,
  'F1_macro': 0.8061988502341788,
  'F1_weighted': 0.863498855490817},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_50',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.306,
  'F1_macro': 0.2221763453272557,
  'F1_weighted': 0.307422497618017},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_100',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.352,
  'F1_macro': 0.2641248112985211,
  'F1_weighted': 0.3556936975581032},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_200',
  'Classifier': 'DecisionTreeClassifier',
  'F1_micro': 0.3955,
  'F1_macro': 0.3048929316628195,
  'F1_weighted': 0.39832857921695636},
 {'Dataset': 'emotion',
  'Preprocessing': 'tokenization',
  'Vectorizer': 'LSA_300',
  'Classifier': 'DecisionTreeCla

### Выводы

1) На датасете emotion
- LSA во всех экспериментах приводила к резкому падению F1‑меры по сравнению с полной TF‑IDF матрицей. Даже при 500 компонентах качество оставалось на 20–30 процентных пунктов ниже исходного. Это объясняется тем, что в датасете эмоций относительно мало документов и классов, а понижение размерности удаляет важные различительные признаки.

2) На датасете news
Результаты более разнообразны:

- Для RandomForestClassifier LSA ухудшала качество, но падение было не столь катастрофичным (с ~0,767 до ~0,738 при 300 компонентах).

- Для DecisionTreeClassifier при использовании LSA качество оставалось на том же уровне (0,616–0,618), а в одном случае (500 компонент) даже немного превышало tf-idf

- Для AdaBoostClassifier LSA улучшала результаты: на lemmatization + TF‑IDF базовая микро‑F1 = 0,6534, а после LSA с 200 компонентами – 0,6618; на lemmatization_noun_adj с 50 компонентами – 0,6534 против исходных 0,6413.

3) Общие мысли
- LSA может быть полезным инструментом для ускорения обучения и уменьшения памяти, особенно когда задача допускает некоторое снижение точности или когда исходная размерность чрезмерно велика.

- На небольших наборах данных или в задачах с малым числом классов (как emotion) применение LSA нецелесообразно, так как потеря информации перевешивает выгоду от уменьшения размерности.

- В более сложных задачах с большим объёмом текста (как 20 newsgroups) LSA может сохранить качество или даже улучшить его для некоторых классификаторов, особенно для ансамблевых методов (AdaBoost).